In [ ]:
!pip install category_encoders

In [ ]:
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

import category_encoders as ce


In [ ]:
df = pd.read_csv("/content/bowler_match_final_stage2.csv")

# ensure date is datetime
df["date"] = pd.to_datetime(df["date"])

print("STAGE 1 ✅ Dataset loaded")
print("➡ Rows:", df.shape[0])
print("➡ Columns:", df.shape[1])
print("➡ Date range:", df["date"].min(), "to", df["date"].max())



STAGE 1 ✅ Dataset loaded
➡ Rows: 9283
➡ Columns: 25
➡ Date range: 2008-04-18 00:00:00 to 2021-04-23 00:00:00


In [ ]:
df.columns

Index(['matchid', 'date', 'season', 'venue', 'city', 'bowling_team',
       'batting_team', 'bowler', 'runs_conceded', 'balls_bowled', 'wicket',
       'wides', 'no_balls', 'overs', 'economy', 'avg_wkts_last_5',
       'avg_wkts_last_10', 'avg_wkts_at_venue', 'matches_at_venue',
       'matches_played', 'career_avg_wickets', 'career_avg_runs_conceded',
       'career_avg_overs', 'discipline_score', 'recent_vs_career_form'],
      dtype='object')

In [ ]:
TARGET = "wicket"

DROP_COLS = [
    "wicket",     # target
    "bowler",     # identifier
    "matchid",    # identifier
    "date"        # time split only
]

X = df.drop(columns=DROP_COLS)
y = df[TARGET]

print("\nSTAGE 2 ✅ Feature/Target separation complete")
print("➡ X shape:", X.shape)
print("➡ y shape:", y.shape)



STAGE 2 ✅ Feature/Target separation complete
➡ X shape: (9283, 21)
➡ y shape: (9283,)


In [ ]:
split_date = df["date"].quantile(0.8)

X_train = X[df["date"] <= split_date].copy()
X_test  = X[df["date"] > split_date].copy()

y_train = y[df["date"] <= split_date].copy()
y_test  = y[df["date"] > split_date].copy()

print("\nSTAGE 3 ✅ Time-based split complete")
print("➡ Train samples:", X_train.shape[0])
print("➡ Test samples:", X_test.shape[0])

assert X_test.shape[0] > 0, "❌ Test set is empty — check season split!"
print("✔ Validation passed: Non-empty test set")


STAGE 3 ✅ Time-based split complete
➡ Train samples: 7434
➡ Test samples: 1849
✔ Validation passed: Non-empty test set


In [ ]:
categorical_cols = [
    "venue",
    "city",
    "bowling_team",
    "batting_team"
]

numerical_cols = [
    "season",
    "runs_conceded",
    "balls_bowled",
    "wides",
    "no_balls",
    "overs",
    "economy",
    "avg_wkts_last_5",
    "avg_wkts_last_10",
    "avg_wkts_at_venue",
    "matches_at_venue",
    "matches_played",
    "career_avg_wickets",
    "career_avg_runs_conceded",
    "career_avg_overs",
    "discipline_score",
    "recent_vs_career_form"
]

print("\nSTAGE 4 ✅ Column grouping complete")
print("➡ Categorical columns:", len(categorical_cols))
print("➡ Numerical columns:", len(numerical_cols))



STAGE 4 ✅ Column grouping complete
➡ Categorical columns: 4
➡ Numerical columns: 17


In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", ce.TargetEncoder(), categorical_cols),
        ("num", StandardScaler(), numerical_cols)
    ],
    remainder="drop"
)

feature_pipeline = Pipeline(
    steps=[
        ("preprocessing", preprocessor)
    ]
)

print("\nSTAGE 5 ✅ Preprocessing pipeline created")



STAGE 5 ✅ Preprocessing pipeline created


In [ ]:
X_train['season'] = X_train['season'].apply(lambda x: int(str(x).split('/')[0]))

feature_pipeline.fit(X_train, y_train)

print("\nSTAGE 6 ✅ Pipeline fitted on TRAIN data only")


STAGE 6 ✅ Pipeline fitted on TRAIN data only


In [ ]:
X_train_transformed = feature_pipeline.transform(X_train)
X_test['season'] = X_test['season'].apply(lambda x: int(str(x).split('/')[0]))
X_test_transformed  = feature_pipeline.transform(X_test)

print("\nSTAGE 7 ✅ Data transformed successfully")
print("➡ Train transformed shape:", X_train_transformed.shape)
print("➡ Test transformed shape:", X_test_transformed.shape)

assert X_train_transformed.shape[1] == X_test_transformed.shape[1], \
       "❌ Feature mismatch between train and test!"
print("✔ Validation passed: Feature dimensions consistent")


STAGE 7 ✅ Data transformed successfully
➡ Train transformed shape: (7434, 21)
➡ Test transformed shape: (1849, 21)
✔ Validation passed: Feature dimensions consistent


In [ ]:
joblib.dump(feature_pipeline, "bowler_feature_pipeline.pkl")

print("\nSTAGE 8 ✅ Bowler feature pipeline saved")
print("➡ File: Bowler_Feature_Pipeline.pkl")


STAGE 8 ✅ Bowler feature pipeline saved
➡ File: Bowler_Feature_Pipeline.pkl
